# Appendix G — Final trio paper figures

This appendix displays the same three validated strategies throughout: buy-and-hold, the AI macro-factor portfolio, and the SJM de-risk overlay. It does not acquire data, choose a strategy, or recalculate financial metrics.

**SJM means Sparse Jump Model.** The SJM overlay can reduce risky-asset exposure in the AI macro-factor portfolio. It does not select a different return-seeking portfolio.

## How to read the four panels

1. **Growth versus total risk:** moving right means more total volatility; moving up means higher compound annual growth.
2. **Market-adjusted return versus residual risk:** moving right means more residual volatility; moving up means a larger market-model intercept.
3. **Ratio comparison:** each row has its own scale. Compare strategies within a row, not across rows.
4. **Relative metric profile:** every row is rescaled separately. Farther right is more favorable for that metric.

The gray reference rays are lines of constant displayed return-to-risk ratio. They are not portfolios, efficient frontiers, capital-allocation lines, or reported Sharpe ratios.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import struct
import sys
from pathlib import Path
from typing import Mapping

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from IPython.display import Markdown, display


DEFAULT_REPORT_RELATIVE = Path(
    "data/provisional_remediation/canonical_reports_devstartfix_full_20260730T154855Z"
)
TABLE_STEM = "tear_sheet_trio_ext2026"
TABLE_RELATIVE = Path("tables") / f"{TABLE_STEM}.parquet"
PAPER_DPI = 300
PNG_SIGNATURE = b"\x89PNG\r\n\x1a\n"

# Fixed, tested categorical palette. It passes the dataviz validator in light mode:
# node .../validate_palette.js '#0072B2,#D55E00,#009E73' --mode light
SURFACE, INK, INK2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE = "#e1e0d9", "#c3c2b7"
PORTFOLIO_STYLE = {
    "static_bh_25pct_2019-01-02": {
        "label": "Buy-and-hold", "short_label": "BH", "color": "#0072B2", "marker": "o"
    },
    "factor_pit_ext2026": {
        "label": "AI macro-factor", "short_label": "AI", "color": "#D55E00", "marker": "s"
    },
}


def _repository_root() -> Path:
    override = os.environ.get("FINANCE_NOTEBOOK_REPO_ROOT")
    if override:
        root = Path(override).expanduser().resolve()
        if not (root / "pyproject.toml").is_file() or not (root / "notebooks").is_dir():
            raise ValueError("FINANCE_NOTEBOOK_REPO_ROOT must contain pyproject.toml and notebooks/")
        return root
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise ValueError("set FINANCE_NOTEBOOK_REPO_ROOT when execution is outside the repository")


def _resolve_path(value: str) -> Path:
    path = Path(value).expanduser()
    return (path if path.is_absolute() else REPO / path).resolve()


def _repo_relative(path: Path, label: str) -> str:
    try:
        return path.resolve().relative_to(REPO.resolve()).as_posix()
    except ValueError as exc:
        raise ValueError(f"{label} must remain inside repository root {REPO}: {path}") from exc


def _report_root() -> Path:
    value = (
        os.environ.get("APPENDIX_G_REPORT_BUNDLE")
        or os.environ.get("FINANCE_NOTEBOOK_REPORT_ROOT")
        or os.environ.get("FINANCE_NOTEBOOK_SOURCE_ROOT")
    )
    return _resolve_path(value) if value else (REPO / DEFAULT_REPORT_RELATIVE).resolve()


def _output_dir() -> Path:
    value = os.environ.get("APPENDIX_G_OUTPUT_DIR") or os.environ.get("FINANCE_NOTEBOOK_OUTPUT_DIR")
    return _resolve_path(value) if value else (REPO / "data" / "appendix_g_final_trio_paper_figures").resolve()


def _sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _require_sha256(value: object, label: str) -> str:
    if not isinstance(value, str) or len(value) != 64 or any(ch not in "0123456789abcdef" for ch in value):
        raise ValueError(f"{label} must be a lowercase SHA-256 digest")
    return value


def _safe_table_path(root: Path, value: object) -> Path:
    if not isinstance(value, str) or chr(92) in value:
        raise ValueError("canonical table path must be a forward-slash relative path")
    relative = Path(value)
    if relative.is_absolute() or ".." in relative.parts or relative != TABLE_RELATIVE:
        raise ValueError(f"canonical table path must be {TABLE_RELATIVE.as_posix()!r}")
    candidate = (root / relative).resolve()
    if root.resolve() not in candidate.parents:
        raise ValueError("canonical table path escapes its report bundle")
    return candidate


def _completed_manifest_hash(root: Path) -> tuple[dict[str, object], str]:
    manifest_path = root / "manifest.json"
    completed_path = root / "COMPLETED"
    if not manifest_path.is_file() or not completed_path.is_file():
        raise ValueError("report bundle requires manifest.json and COMPLETED")
    manifest_bytes = manifest_path.read_bytes()
    manifest_sha256 = hashlib.sha256(manifest_bytes).hexdigest()
    marker_lines = completed_path.read_text(encoding="utf-8").splitlines()
    if marker_lines != [f"manifest_sha256={manifest_sha256}"]:
        raise ValueError("COMPLETED does not bind the exact report manifest bytes")
    try:
        manifest = json.loads(manifest_bytes)
    except json.JSONDecodeError as exc:
        raise ValueError("report manifest is not valid JSON") from exc
    if not isinstance(manifest, dict) or manifest.get("completed") is not True:
        raise ValueError("report manifest must be an object with completed=true")
    return manifest, manifest_sha256


def load_completed_trio() -> tuple[pd.DataFrame, dict[str, object], dict[str, object], str, str]:
    """Strictly load the one canonical trio projection; no metrics are recomputed."""
    if not REPORT_ROOT.is_dir():
        raise ValueError(f"report bundle is absent or not a directory: {REPORT_ROOT}")
    if str(REPO) not in sys.path:
        sys.path.insert(0, str(REPO))

    # This producer-owned validator checks the complete table/mirror family, manifest
    # input lineage, row contracts, inventory containment, hashes, and COMPLETED marker.
    from scripts.build_tear_sheet import validate_canonical_report_bundle

    validate_canonical_report_bundle(REPORT_ROOT)
    manifest, manifest_sha256 = _completed_manifest_hash(REPORT_ROOT)
    if manifest.get("schema") != "canonical_reports.v1":
        raise ValueError("unexpected canonical report manifest schema")
    if manifest.get("producer") != "scripts/build_tear_sheet.py":
        raise ValueError("unexpected report manifest producer")

    input_manifests = manifest.get("input_manifests")
    if not isinstance(input_manifests, Mapping) or set(input_manifests) != {"factor_run", "sjm_run", "market_snapshot"}:
        raise ValueError("report manifest must pin factor, SJM, and market input manifests")
    for name, identity_key in (("factor_run", "run_id"), ("sjm_run", "run_id"), ("market_snapshot", "snapshot_id")):
        entry = input_manifests[name]
        if not isinstance(entry, Mapping) or not isinstance(entry.get(identity_key), str):
            raise ValueError(f"input_manifests.{name} lacks {identity_key}")
        _require_sha256(entry.get("manifest_sha256"), f"input_manifests.{name}.manifest_sha256")

    tables = manifest.get("tables")
    if not isinstance(tables, Mapping) or TABLE_STEM not in tables:
        raise ValueError(f"report manifest lacks {TABLE_STEM}")
    entry = tables[TABLE_STEM]
    if not isinstance(entry, Mapping):
        raise ValueError(f"{TABLE_STEM} manifest entry must be an object")
    table_path = _safe_table_path(REPORT_ROOT, entry.get("file"))
    table_sha256 = _require_sha256(entry.get("sha256"), f"tables.{TABLE_STEM}.sha256")
    if entry.get("schema") != "tear_sheet.trio.v4" or entry.get("rows") != 3:
        raise ValueError(f"{TABLE_STEM} inventory must be the three-row tear_sheet.trio.v4")
    if not table_path.is_file() or _sha256_file(table_path) != table_sha256:
        raise ValueError(f"{TABLE_STEM} is missing or differs from its manifest hash")

    trio = pd.read_parquet(table_path)
    if len(trio) != entry["rows"]:
        raise ValueError(f"{TABLE_STEM} row count differs from manifest inventory")
    required = {
        "schema", "portfolio_id", "start", "end", "n_obs", "periods_per_year",
        "cash_benchmark_id", "currency_basis", "source", "row_kind", "ann_vol",
        "cagr", "calmar", "maxdd", "sharpe", "raw_market_model_intercept_ann_arithmetic",
        "raw_market_model_r2", "raw_market_model_start", "raw_market_model_end",
        "raw_market_model_n_obs", "raw_market_model_kind", "ssr_ssr",
    }
    missing = sorted(required - set(trio.columns))
    if missing:
        raise ValueError(f"{TABLE_STEM} is missing required display/provenance columns: {missing}")
    if set(trio["schema"]) != {"portfolio_metrics.reader.v2"} or set(trio["row_kind"]) != {"full"}:
        raise ValueError("trio rows must be full portfolio_metrics.reader.v2 records")
    if trio["portfolio_id"].duplicated().any():
        raise ValueError("trio portfolio identifiers must be unique")

    factor = trio[trio["portfolio_id"] == "factor_pit_ext2026"]
    sjm_run_id = input_manifests["sjm_run"]["run_id"]
    sjm = trio[trio["portfolio_id"].astype(str) == sjm_run_id]
    static = trio[trio["portfolio_id"].astype(str).str.startswith("static_bh_")]
    if len(factor) != 1 or len(sjm) != 1 or len(static) != 1 or len(trio) != 3:
        raise ValueError("trio must contain exactly one static, factor_pit_ext2026, and pinned SJM row")
    if not str(factor.iloc[0]["source"]).startswith("scripts/extend_stream_2026.py:"):
        raise ValueError("factor trio provenance must be the extended-stream source")
    if not str(sjm.iloc[0]["source"]).startswith(f"sjm_run:{sjm_run_id}/"):
        raise ValueError("SJM trio provenance does not bind the pinned SJM run")
    if not str(static.iloc[0]["source"]).startswith("scripts/build_tear_sheet.py:"):
        raise ValueError("static trio provenance is not the canonical report producer")

    shared = trio[["start", "end", "n_obs", "periods_per_year", "cash_benchmark_id", "currency_basis"]].drop_duplicates()
    if len(shared) != 1:
        raise ValueError("trio rows must share one performance window and financial convention")
    attribution = trio[["raw_market_model_start", "raw_market_model_end", "raw_market_model_n_obs"]].copy()
    attribution.columns = ["start", "end", "n_obs"]
    expected_attribution = shared.iloc[0][["start", "end", "n_obs"]]
    if not (attribution["start"].eq(expected_attribution["start"]).all()
            and attribution["end"].eq(expected_attribution["end"]).all()
            and attribution["n_obs"].eq(expected_attribution["n_obs"]).all()):
        raise ValueError("full-row raw-market-model attribution must match the performance window")
    if set(trio["raw_market_model_kind"]) != {"raw_market_model"}:
        raise ValueError("trio attribution must retain its raw-market-model label")
    return trio, manifest, dict(input_manifests), manifest_sha256, table_sha256


REPO = _repository_root()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from macro_framework.appendix_presentation import draw_reference_rays
REPORT_ROOT = _report_root()
OUTPUT_DIR = _output_dir()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trio_raw, report_manifest, input_manifests, REPORT_MANIFEST_SHA256, CANONICAL_TABLE_SHA256 = load_completed_trio()
print(f"validated report bundle: {REPORT_ROOT}")
print(f"validated canonical table: {TABLE_RELATIVE} #{CANONICAL_TABLE_SHA256}")
print(f"presentation output namespace: {OUTPUT_DIR}")

validated report bundle: /home/mc/projects/Global_Macro_AI_Factors/data/provisional_remediation/canonical_reports_devstartfix_full_20260730T154855Z
validated canonical table: tables/tear_sheet_trio_ext2026.parquet #1380f9aa2ed77e7b86154d95407e0b9fbd32d92a52f6b02af450310e780497cf
presentation output namespace: /home/mc/projects/Global_Macro_AI_Factors/data/appendix_g_final_trio_paper_figures


In [2]:
SJM_PORTFOLIO_ID = input_manifests["sjm_run"]["run_id"]
PORTFOLIO_STYLE[SJM_PORTFOLIO_ID] = {
    "label": "SJM de-risk", "short_label": "SJM", "color": "#009E73", "marker": "^"
}
PORTFOLIO_ORDER = ["static_bh_25pct_2019-01-02", "factor_pit_ext2026", SJM_PORTFOLIO_ID]


def canonical_trio_view(table: pd.DataFrame) -> pd.DataFrame:
    """Presentation projection only; canonical fields remain unchanged in trio_raw."""
    view = table.set_index("portfolio_id").loc[PORTFOLIO_ORDER].reset_index().copy()
    numeric_columns = [
        "ann_vol", "cagr", "calmar", "maxdd", "sharpe",
        "raw_market_model_intercept_ann_arithmetic", "raw_market_model_r2",
    ]
    for column in numeric_columns:
        view[column] = pd.to_numeric(view[column], errors="raise")
        if not np.isfinite(view[column]).all():
            raise ValueError(f"non-finite canonical display field: {column}")
    if (view["ann_vol"] <= 0).any():
        raise ValueError("annualized volatility must be positive for the display projection")
    if ((view["raw_market_model_r2"] < 0) | (view["raw_market_model_r2"] >= 1)).any():
        raise ValueError("raw-market-model R² must be in [0, 1) for residual-risk display")

    # These are presentation-only transformations of canonical display fields.
    view["residual_vol_ann"] = view["ann_vol"] * np.sqrt(1.0 - view["raw_market_model_r2"])
    if (view["residual_vol_ann"] <= 0).any() or not np.isfinite(view["residual_vol_ann"]).all():
        raise ValueError("residual volatility must be finite and positive")
    view["appraisal"] = view["raw_market_model_intercept_ann_arithmetic"] / view["residual_vol_ann"]
    view["label"] = view["portfolio_id"].map(lambda value: PORTFOLIO_STYLE[value]["label"])
    view["short_label"] = view["portfolio_id"].map(lambda value: PORTFOLIO_STYLE[value]["short_label"])
    view["color"] = view["portfolio_id"].map(lambda value: PORTFOLIO_STYLE[value]["color"])
    view["marker"] = view["portfolio_id"].map(lambda value: PORTFOLIO_STYLE[value]["marker"])
    return view


trio_view = canonical_trio_view(trio_raw)
display_table = pd.DataFrame({
    "Strategy": trio_view["label"],
    "CAGR": trio_view["cagr"].map(lambda value: f"{value:.1%}"),
    "Total volatility (annualized)": trio_view["ann_vol"].map(lambda value: f"{value:.1%}"),
    "Market-model intercept (annualized)": trio_view["raw_market_model_intercept_ann_arithmetic"].map(lambda value: f"{value:.1%}"),
    "Residual volatility (annualized)": trio_view["residual_vol_ann"].map(lambda value: f"{value:.1%}"),
    "Sharpe": trio_view["sharpe"].map(lambda value: f"{value:.2f}"),
    "Calmar": trio_view["calmar"].map(lambda value: f"{value:.2f}"),
    "Derived appraisal ratio": trio_view["appraisal"].map(lambda value: f"{value:.2f}"),
    "Max drawdown": trio_view["maxdd"].map(lambda value: f"{value:.1%}"),
})
print("Appendix G display table — values are formatted projections of canonical fields:")
display(display_table)
display(Markdown(
    "**Metric guide.** CAGR is compound annual growth. Sharpe uses the validated cash-excess return stream, "
    "so it is not CAGR divided by volatility. Calmar is CAGR divided by the absolute maximum drawdown. "
    "Residual volatility is total volatility multiplied by the square root of one minus market-model R-squared. "
    "The derived appraisal ratio is the market-model intercept divided by residual volatility."
))
print(
    f"Common canonical window: {trio_view.iloc[0]['start']} to {trio_view.iloc[0]['end']} | "
    f"n={trio_view.iloc[0]['n_obs']} | {trio_view.iloc[0]['periods_per_year']} periods/year"
)

Appendix G display table — values are formatted projections of canonical fields:


,Strategy,CAGR,Total volatility (annualized),Market-model intercept (annualized),Residual volatility (annualized),Sharpe,Calmar,Derived appraisal ratio,Max drawdown
0,Buy-and-hold,17.8%,13.6%,7.1%,7.8%,1.10,0.99,0.92,-17.8%
1,AI macro-factor,12.8%,9.3%,8.5%,8.1%,1.09,1.12,1.05,-11.4%
2,SJM de-risk,11.2%,7.9%,8.0%,7.1%,1.08,1.27,1.12,-8.8%


**Metric guide.** CAGR is compound annual growth. Sharpe uses the validated cash-excess return stream, so it is not CAGR divided by volatility. Calmar is CAGR divided by the absolute maximum drawdown. Residual volatility is total volatility multiplied by the square root of one minus market-model R-squared. The derived appraisal ratio is the market-model intercept divided by residual volatility.

Common canonical window: 2019-01-03 00:00:00 to 2026-06-30 00:00:00 | n=1845 | 252 periods/year


## Figure reading notes

- The colored marks identify the same strategy in every panel. The marker shape provides a second cue for print and color-vision accessibility.
- In the two maps, a steeper gray ray means a higher displayed ratio. The rays are reference lines only, not investable portfolios or frontiers.
- In the ratio ladder, each row runs from that metric's lowest observed value to its highest observed value. Compare positions only within the same row.
- In the metric profile, each metric is rescaled separately. Lower residual volatility is treated as more favorable, and a less-negative drawdown is better. Horizontal distances cannot be compared across rows.

In [3]:
mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titleweight": "semibold",
    "axes.labelcolor": INK2,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
})


def _style_axis(ax: plt.Axes, *, grid_axis: str = "both") -> None:
    ax.set_facecolor(SURFACE)
    ax.set_axisbelow(True)
    if grid_axis in {"x", "y", "both"}:
        ax.grid(axis=grid_axis, color=GRID, linewidth=0.55)
    ax.tick_params(labelsize=7, length=2.5, color=BASELINE)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_color(BASELINE)
    ax.spines["bottom"].set_color(BASELINE)


def _legend_handles() -> list[Line2D]:
    return [
        Line2D(
            [], [], color=PORTFOLIO_STYLE[row.portfolio_id]["color"], marker=PORTFOLIO_STYLE[row.portfolio_id]["marker"],
            linestyle="None", markeredgecolor=SURFACE, markeredgewidth=0.9, markersize=6,
            label=PORTFOLIO_STYLE[row.portfolio_id]["label"],
        )
        for row in trio_view.itertuples()
    ]


def _scatter_rows(ax: plt.Axes, x: str, y: str, offsets: Mapping[str, tuple[float, float]]) -> None:
    for row in trio_view.itertuples():
        ax.scatter(
            getattr(row, x), getattr(row, y), s=44, marker=row.marker, color=row.color,
            edgecolors=SURFACE, linewidths=1.1, zorder=3,
        )
        dx, dy = offsets[row.portfolio_id]
        ax.annotate(
            row.short_label, (getattr(row, x), getattr(row, y)), xytext=(dx, dy), textcoords="offset points",
            fontsize=6.7, color=INK, weight="semibold", zorder=4,
        )



def draw_total_risk_map(ax: plt.Axes, *, legend: bool = True) -> None:
    x_max = float(trio_view["ann_vol"].max() * 1.34)
    y_max = float(trio_view["cagr"].max() * 1.34)
    _style_axis(ax)
    draw_reference_rays(ax, (0.8, 1.1, 1.4), x_max=x_max, y_max=y_max, color=BASELINE)
    _scatter_rows(
        ax, "ann_vol", "cagr",
        {"static_bh_25pct_2019-01-02": (4, 4), "factor_pit_ext2026": (4, -11), SJM_PORTFOLIO_ID: (4, 4)},
    )
    ax.set(xlim=(0, x_max), ylim=(0, y_max), xlabel="Annualized volatility (total risk)", ylabel="Compound annual growth rate")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
    ax.set_title("Growth versus total volatility", loc="left", color=INK, fontsize=9, y=1.095, pad=0)
    ax.text(0.0, 1.035, "Gray rays: constant CAGR divided by volatility; not Sharpe ratios or a frontier", transform=ax.transAxes,
            fontsize=5.8, color=INK2, va="bottom")
    if legend:
        ax.legend(handles=_legend_handles(), loc="upper left", frameon=False, fontsize=5.8,
                  labelcolor=INK2, borderpad=0.1, handletextpad=0.35)


def draw_regression_adjusted_map(ax: plt.Axes, *, legend: bool = True) -> None:
    x_max = float(trio_view["residual_vol_ann"].max() * 1.38)
    y_max = float(trio_view["raw_market_model_intercept_ann_arithmetic"].max() * 1.38)
    _style_axis(ax)
    draw_reference_rays(ax, (0.8, 1.0, 1.2), x_max=x_max, y_max=y_max, color=BASELINE)
    _scatter_rows(
        ax, "residual_vol_ann", "raw_market_model_intercept_ann_arithmetic",
        {"static_bh_25pct_2019-01-02": (4, -11), "factor_pit_ext2026": (4, 4), SJM_PORTFOLIO_ID: (-24, 4)},
    )
    ax.set(xlim=(0, x_max), ylim=(0, y_max), xlabel="Residual volatility", ylabel="Annualized market-model intercept")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
    ax.set_title("Market-adjusted return versus residual risk", loc="left", color=INK, fontsize=9, y=1.095, pad=0)
    ax.text(0.0, 1.035, "Gray rays: intercept divided by residual volatility; derived appraisal ratio", transform=ax.transAxes,
            fontsize=5.8, color=INK2, va="bottom")
    if legend:
        ax.legend(handles=_legend_handles(), loc="upper left", frameon=False, fontsize=5.8,
                  labelcolor=INK2, borderpad=0.1, handletextpad=0.35)


def draw_ratio_ladder(ax: plt.Axes, *, legend: bool = True) -> None:
    _style_axis(ax, grid_axis="none")
    metrics = (("Sharpe", "sharpe"), ("Calmar", "calmar"), ("Appraisal", "appraisal"))
    y_positions = np.arange(len(metrics))[::-1]
    dodge = (-0.16, 0.0, 0.16)
    for y, (label, column) in zip(y_positions, metrics):
        values = trio_view[column].to_numpy(dtype=float)
        lo, hi = float(values.min()), float(values.max())
        span = hi - lo
        positions = np.full_like(values, 0.5) if span == 0 else (values - lo) / span
        ax.hlines(y, 0, 1, color=BASELINE, linewidth=1.0, zorder=0)
        ax.vlines((0, 1), y - 0.075, y + 0.075, color=BASELINE, linewidth=0.8)
        for row, position, y_offset in zip(trio_view.itertuples(), positions, dodge):
            ax.scatter(position, y + y_offset, s=39, marker=row.marker, color=row.color,
                       edgecolors=SURFACE, linewidths=1.0, zorder=3)
            ax.annotate(f"{getattr(row, column):.2f}", (position, y + y_offset), xytext=(4, 0),
                        textcoords="offset points", va="center", fontsize=5.8, color=INK)
    ax.set(ylim=(-0.55, len(metrics) - 0.35), xlim=(-0.14, 1.22), yticks=y_positions, yticklabels=[m[0] for m in metrics])
    ax.set_xticks([])
    ax.set_title("Within-metric ratio comparison", loc="left", color=INK, fontsize=9, y=1.095, pad=0)
    ax.text(0.0, 1.035, "Each row has its own minimum-to-maximum scale; compare within rows only",
            transform=ax.transAxes, fontsize=5.8, color=INK2, va="bottom")
    if legend:
        ax.legend(handles=_legend_handles(), loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3,
                  frameon=False, fontsize=4.5, labelcolor=INK2, borderpad=0.1, handletextpad=0.25,
                  columnspacing=0.6)


def draw_metric_profile(ax: plt.Axes, *, legend: bool = True) -> None:
    _style_axis(ax, grid_axis="none")
    metrics = (
        ("CAGR", "cagr", False),
        ("Intercept\n(annualized)", "raw_market_model_intercept_ann_arithmetic", False),
        ("Max drawdown", "maxdd", False),
        ("Residual vol.", "residual_vol_ann", True),
        ("Sharpe", "sharpe", False),
        ("Appraisal", "appraisal", False),
    )
    y_positions = np.arange(len(metrics))[::-1]
    dodge = (-0.15, 0.0, 0.15)
    for y, (label, column, invert) in zip(y_positions, metrics):
        values = trio_view[column].to_numpy(dtype=float)
        favorable = -values if invert else values
        lo, hi = float(favorable.min()), float(favorable.max())
        spread = hi - lo
        normalized = np.full_like(favorable, 0.5) if spread == 0 else (favorable - lo) / spread
        relative_spread = spread / max(float(np.median(np.abs(favorable))), 1e-12)
        damping = min(1.0, 0.25 + 0.75 * relative_spread)
        positions = 0.5 + (normalized - 0.5) * damping
        ax.hlines(y, 0, 1, color=BASELINE, linewidth=0.9, zorder=0)
        for row, position, y_offset in zip(trio_view.itertuples(), positions, dodge):
            ax.scatter(position, y + y_offset, s=36, marker=row.marker, color=row.color,
                       edgecolors=SURFACE, linewidths=1.0, zorder=3)
    ax.set(ylim=(-0.6, len(metrics) - 0.35), xlim=(-0.03, 1.03), yticks=y_positions, yticklabels=["CAGR", "Market-model\nintercept", "Max drawdown", "Residual volatility\n(lower is better)", "Sharpe", "Appraisal"])
    ax.set_xticks((0, 0.5, 1), ("less favorable", "middle", "more favorable"))
    ax.set_title("Relative metric profile", loc="left", color=INK, fontsize=9, y=1.095, pad=0)
    ax.text(0.0, 1.035, "Each metric is rescaled separately; farther right is more favorable",
            transform=ax.transAxes, fontsize=5.8, color=INK2, va="bottom")
    if legend:
        ax.legend(handles=_legend_handles(), loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3,
                  frameon=False, fontsize=4.5, labelcolor=INK2, borderpad=0.1, handletextpad=0.25,
                  columnspacing=0.6)


def _save_standalone(filename: str, height_inches: float, draw) -> Path:
    path = OUTPUT_DIR / filename
    figure, axis = plt.subplots(figsize=(3.35, height_inches), dpi=PAPER_DPI, layout="constrained")
    figure.patch.set_facecolor(SURFACE)
    draw(axis, legend=True)
    figure.savefig(path, dpi=PAPER_DPI, facecolor=SURFACE)
    plt.close(figure)
    return path


standalone_paths = [
    _save_standalone("appendix_g_total_risk_map.png", 2.60, draw_total_risk_map),
    _save_standalone("appendix_g_regression_adjusted_map.png", 2.60, draw_regression_adjusted_map),
    _save_standalone("appendix_g_ratio_ladder.png", 3.15, draw_ratio_ladder),
    _save_standalone("appendix_g_metric_profile.png", 3.70, draw_metric_profile),
]

composite_path = OUTPUT_DIR / "appendix_g_final_trio_composite.png"
composite, composite_axes = plt.subplots(4, 1, figsize=(6.90, 10.60), dpi=PAPER_DPI)
composite.patch.set_facecolor(SURFACE)
draw_total_risk_map(composite_axes[0], legend=False)
draw_regression_adjusted_map(composite_axes[1], legend=False)
draw_ratio_ladder(composite_axes[2], legend=False)
draw_metric_profile(composite_axes[3], legend=False)
composite.suptitle("Appendix G — Final trio paper figures", x=0.01, ha="left", y=0.992, fontsize=12, color=INK, weight="semibold")
composite.legend(handles=_legend_handles(), loc="upper center", bbox_to_anchor=(0.5, 0.974), ncol=3,
                 frameon=False, fontsize=7.2, labelcolor=INK2, handletextpad=0.35, columnspacing=1.2)
composite.subplots_adjust(left=0.15, right=0.92, top=0.91, bottom=0.05, hspace=0.76)
composite.savefig(composite_path, dpi=PAPER_DPI, facecolor=SURFACE)
plt.close(composite)

print("Exported paper figures:")
for path in [*standalone_paths, composite_path]:
    print(path)

Exported paper figures:
/home/mc/projects/Global_Macro_AI_Factors/data/appendix_g_final_trio_paper_figures/appendix_g_total_risk_map.png
/home/mc/projects/Global_Macro_AI_Factors/data/appendix_g_final_trio_paper_figures/appendix_g_regression_adjusted_map.png
/home/mc/projects/Global_Macro_AI_Factors/data/appendix_g_final_trio_paper_figures/appendix_g_ratio_ladder.png
/home/mc/projects/Global_Macro_AI_Factors/data/appendix_g_final_trio_paper_figures/appendix_g_metric_profile.png
/home/mc/projects/Global_Macro_AI_Factors/data/appendix_g_final_trio_paper_figures/appendix_g_final_trio_composite.png


In [4]:
def _png_dimensions(path: Path) -> tuple[int, int]:
    payload = path.read_bytes()
    if payload[:8] != PNG_SIGNATURE or payload[12:16] != b"IHDR":
        raise ValueError(f"not a valid PNG signature/IHDR sequence: {path}")
    return struct.unpack(">II", payload[16:24])


def _notebook_source_path() -> Path:
    value = os.environ.get("APPENDIX_G_NOTEBOOK_PATH")
    return _resolve_path(value) if value else (REPO / "notebooks" / "appendix_g_final_trio_paper_figures.ipynb").resolve()


def _guard_existing_presentation_manifest(path: Path) -> None:
    if not path.exists():
        return
    try:
        prior = json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        raise ValueError(f"existing presentation manifest is invalid: {path}") from exc
    projection = prior.get("projection_of", {}) if isinstance(prior, dict) else {}
    if projection.get("report_manifest_sha256") != REPORT_MANIFEST_SHA256 or projection.get("canonical_table_sha256") != CANONICAL_TABLE_SHA256:
        raise ValueError("refusing to overwrite Appendix G artifacts derived from a different canonical source")


all_paths = [*standalone_paths, composite_path]
output_inventory = {}
for path in all_paths:
    width_px, height_px = _png_dimensions(path)
    output_inventory[path.name] = {
        "sha256": _sha256_file(path),
        "media_type": "image/png",
        "width_px": width_px,
        "height_px": height_px,
        "dpi": PAPER_DPI,
        "width_in": round(width_px / PAPER_DPI, 4),
        "height_in": round(height_px / PAPER_DPI, 4),
    }
    print(f"{path.name}: PNG signature OK, {width_px}×{height_px} px ({width_px / PAPER_DPI:.2f}×{height_px / PAPER_DPI:.2f} in)")

notebook_path = _notebook_source_path()
if not notebook_path.is_file():
    raise ValueError(f"Appendix G notebook source is absent: {notebook_path}")
presentation_manifest_path = OUTPUT_DIR / "appendix_g_presentation_manifest.json"
_guard_existing_presentation_manifest(presentation_manifest_path)
presentation_manifest = {
    "schema": "appendix_g.presentation_figures.v1",
    "producer": "notebooks/appendix_g_final_trio_paper_figures.ipynb",
    "presentation_only": True,
    "notebook": {
        "file": _repo_relative(notebook_path, "Appendix G notebook"),
        "sha256": _sha256_file(notebook_path),
    },
    "projection_of": {
        "report_bundle": _repo_relative(REPORT_ROOT, "Appendix G report bundle"),
        "report_id": report_manifest["report_id"],
        "report_manifest_sha256": REPORT_MANIFEST_SHA256,
        "canonical_table": TABLE_RELATIVE.as_posix(),
        "canonical_table_schema": "tear_sheet.trio.v4",
        "canonical_table_sha256": CANONICAL_TABLE_SHA256,
        "factor_run": dict(input_manifests["factor_run"]),
        "sjm_run": dict(input_manifests["sjm_run"]),
        "market_snapshot": dict(input_manifests["market_snapshot"]),
    },
    "figure_order": [path.name for path in all_paths],
    "display_table_columns": list(display_table.columns),
    "palette": {
        "validated_with": "dataviz/scripts/validate_palette.js --mode light",
        "categorical_hex": ["#0072B2", "#D55E00", "#009E73"],
        "secondary_encoding": "circle/square/triangle markers",
    },
    "outputs": output_inventory,
}
presentation_manifest_path.write_text(json.dumps(presentation_manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(f"wrote presentation manifest: {presentation_manifest_path}")
pd.DataFrame.from_dict(output_inventory, orient="index")

appendix_g_total_risk_map.png: PNG signature OK, 1005×780 px (3.35×2.60 in)
appendix_g_regression_adjusted_map.png: PNG signature OK, 1005×780 px (3.35×2.60 in)
appendix_g_ratio_ladder.png: PNG signature OK, 1005×945 px (3.35×3.15 in)
appendix_g_metric_profile.png: PNG signature OK, 1005×1110 px (3.35×3.70 in)
appendix_g_final_trio_composite.png: PNG signature OK, 2070×3180 px (6.90×10.60 in)
wrote presentation manifest: /home/mc/projects/Global_Macro_AI_Factors/data/appendix_g_final_trio_paper_figures/appendix_g_presentation_manifest.json


,sha256,media_type,width_px,height_px,dpi,width_in,height_in
appendix_g_total_risk_map.png,fff87b3ca5e4e2f1579d680cb759220cbf4cc600940ab9...,image/png,1005,780,300,3.35,2.60
appendix_g_regression_adjusted_map.png,e31f81a147b45d55a91e7e8965d7c53efb71514cd7b2e1...,image/png,1005,780,300,3.35,2.60
appendix_g_ratio_ladder.png,94ef3d138320d89cb3fcf1db89b8a6cee622af5b80809b...,image/png,1005,945,300,3.35,3.15
appendix_g_metric_profile.png,61442729a84fdbe0b6166638cb617f862984222fc266cf...,image/png,1005,1110,300,3.35,3.70
appendix_g_final_trio_composite.png,078e36b47a3fd79745bc5147b9622c22fb4964d84ae947...,image/png,2070,3180,300,6.90,10.60
